# 01. Exploratory Data Analysis (EDA)
## House Price Prediction System using Machine Learning Algorithms and Visualization

This notebook explores the Kaggle Ames Iowa Housing dataset (1,460 residential properties with 79 explanatory features).
Key objectives:
1. Inspect dataset dimensions, feature types, and statistical properties.
2. Profile target variable (`SalePrice`): skewness, kurtosis, and quartiles.
3. Investigate missing values across columns.
4. Identify outliers (e.g. `GrLivArea > 4000` sq ft).
5. Compute Pearson correlation coefficients and highlight top correlated features.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from src.data_loader import load_raw_data, get_dataset_summary
from src.config import FIGURES_DIR, TARGET_COLUMN
from src.visualize import (
    plot_missing_values,
    plot_target_distribution,
    plot_outliers_before_after,
    plot_correlation_heatmap,
    plot_top_correlated_bar,
    plot_key_features_scatter,
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

### 1. Load Dataset & Statistical Overview

In [ ]:
train_df, test_df = load_raw_data()
print(f"Training set shape: {train_df.shape}")
if test_df is not None:
    print(f"Test set shape: {test_df.shape}")

summary = get_dataset_summary(train_df)
print("\n--- Dataset Summary Profile ---")
for k, v in summary.items():
    if k != 'target_statistics':
        print(f"{k}: {v}")

### 2. Target Variable Analysis (`SalePrice`)

In [ ]:
price = train_df['SalePrice']
print(f"SalePrice Statistics ($ USD):")
print(f"Count:    {price.count()}")
print(f"Mean:     ${price.mean():,.2f}")
print(f"Std:      ${price.std():,.2f}")
print(f"Min:      ${price.min():,.2f}")
print(f"25% (Q1): ${price.quantile(0.25):,.2f}")
print(f"Median:   ${price.median():,.2f}")
print(f"75% (Q3): ${price.quantile(0.75):,.2f}")
print(f"Max:      ${price.max():,.2f}")
print(f"Skewness: {price.skew():.3f}")
print(f"Kurtosis: {price.kurt():.3f}")

# Log-transform comparison
fig_target = plot_target_distribution(price.values, np.log1p(price.values))
plt.show()

### 3. Missing Value Analysis

In [ ]:
fig_missing = plot_missing_values(train_df)
plt.show()

### 4. Outlier Analysis & Filtration

In [ ]:
outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)]
print(f"Identified {len(outliers)} severe outliers with GrLivArea > 4000 sq ft and low SalePrice:")
display(outliers[['Id', 'GrLivArea', 'OverallQual', 'YearBuilt', 'SalePrice']])

cleaned_df = train_df[~((train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000))]
fig_outliers = plot_outliers_before_after(train_df, cleaned_df)
plt.show()

### 5. Correlation Matrix & Top 10 Correlated Features

In [ ]:
fig_corr = plot_correlation_heatmap(cleaned_df, top_k=15)
plt.show()

from src.features import get_top_correlated_features
top_corr = get_top_correlated_features(cleaned_df, top_n=10)
fig_top = plot_top_correlated_bar(top_corr)
plt.show()

### 6. Key Feature Scatter Plots vs. SalePrice

In [ ]:
fig_scatter = plot_key_features_scatter(cleaned_df)
plt.show()